In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
import numpy as np
import os
import json
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
BASE_DIR = os.path.dirname(os.path.abspath(__file__))
dataset_path = os.path.join(BASE_DIR, "hand_gesture_dataset_processed_10")

In [ ]:
train_dir = os.path.join(dataset_path, "train")
val_dir = os.path.join(dataset_path, "val")
test_dir = os.path.join(dataset_path, "test")

In [ ]:
train_data = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=(50, 50),
    batch_size=32,
    color_mode='grayscale',
    label_mode='categorical',
    shuffle=True
)

val_data = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    image_size=(50, 50),
    batch_size=32,
    color_mode='grayscale',
    label_mode='categorical',
    shuffle=False
)

test_data = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=(50, 50),
    batch_size=32,
    color_mode='grayscale',
    label_mode='categorical',
    shuffle=False
)

Found 557 files belonging to 8 classes.
Found 160 files belonging to 8 classes.
Found 80 files belonging to 8 classes.


In [ ]:
class_names = train_data.class_names
print(class_names)
num_classes = len(class_names)
print(f"Number of classes: {num_classes}")

['Background', 'New Speed 1', 'New Speed 2', 'New Speed 3', 'New Speed 4', 'New Stop Images', 'Pointing Left', 'Pointing Right']
Number of classes: 8


In [ ]:
with open(os.path.join(BASE_DIR, "class_indices_v3.json"), "w") as f:
    json.dump(train_data.class_names, f)

model = models.Sequential([
    layers.Rescaling(1./255, input_shape=(50, 50, 1)),  # Normalize pixel values to [0,1]
    layers.Conv2D(16, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(32, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, padding='same', activation='relu'),
    layers.Flatten(),
    layers.Dropout(0.2),
    layers.Dense(128, activation='relu'),
    layers.Dense(num_classes, activation='softmax')
])

In [ ]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 rescaling (Rescaling)       (None, 50, 50, 1)         0         
                                                                 
 conv2d (Conv2D)             (None, 50, 50, 16)        160       
                                                                 
 max_pooling2d (MaxPooling2  (None, 25, 25, 16)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 25, 25, 32)        4640      
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 12, 12, 32)        0         
 g2D)                                                            
                                                                 
 conv2d_2 (Conv2D)           (None, 12, 12, 64)        1

In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
checkpoint_path = os.path.join(BASE_DIR, "hand_gesture_model_best.keras")
checkpoint = ModelCheckpoint(checkpoint_path, monitor='val_loss', save_best_only=True, verbose=1)
earlystop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)
callbacks = [checkpoint, earlystop]

In [ ]:
print("\nTraining the model...")
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=30,
    callbacks=callbacks,
    verbose=1
)


Training the model...
Epoch 1/30


15/18 [========================>.....] - ETA: 0s - loss: 1.6190 - accuracy: 0.3833
Epoch 1: val_loss improved from inf to 0.81674, saving model to c:\Users\louis\Documents\Telerobotics\hand_gesture_model_best.keras
18/18 [==============================] - 2s 39ms/step - loss: 1.5146 - accuracy: 0.4273 - val_loss: 0.8167 - val_accuracy: 0.6687
Epoch 2/30
16/18 [=========================>....] - ETA: 0s - loss: 0.5698 - accuracy: 0.7559
Epoch 2: val_loss improved from 0.81674 to 0.51956, saving model to c:\Users\louis\Documents\Telerobotics\hand_gesture_model_best.keras
18/18 [==============================] - 1s 36ms/step - loss: 0.5795 - accuracy: 0.7594 - val_loss: 0.5196 - val_accuracy: 0.8250
Epoch 3/30
17/18 [===========================>..] - ETA: 0s - loss: 0.5036 - accuracy: 0.8070
Epoch 3: val_loss improved from 0.51956 to 0.42296, saving model to c:\Users\louis\Documents\Telerobotics\hand_gesture_model_best.keras
18/18 [======================

In [ ]:
print("\nEvaluating on test data...")
test_loss, test_accuracy = model.evaluate(test_data)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")


Evaluating on test data...
3/3 [==============================] - 0s 5ms/step - loss: 0.0334 - accuracy: 0.9875
Test Loss: 0.0334
Test Accuracy: 0.9875


In [ ]:
y_true = np.concatenate([np.argmax(y.numpy(), axis=1) for _, y in test_data], axis=0)
preds = model.predict(test_data, verbose=1)
y_pred = np.argmax(preds, axis=1)

# Create directories for saving model and results
os.makedirs(os.path.join(BASE_DIR, "models", "model_10"), exist_ok=True)
os.makedirs(os.path.join(BASE_DIR, "results", "model_10"), exist_ok=True)

# Print and save confusion matrix and classification report
cm = confusion_matrix(y_true, y_pred)
cr = classification_report(y_true, y_pred, target_names=class_names)
print("\nConfusion Matrix:\n", cm)
print("\nClassification Report:\n", cr)
with open(os.path.join(BASE_DIR, "results", "model_10", "classification_report.txt"), "w") as f:
    f.write("Confusion Matrix:\n")
    f.write(str(cm))
    f.write("\n\nClassification Report:\n")
    f.write(cr)

# Plot confusion matrix (raw counts) and normalized version
tick_marks = np.arange(len(class_names))

# Raw confusion matrix heatmap
plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.colorbar()
plt.xticks(tick_marks, class_names, rotation=45, ha='right')
plt.yticks(tick_marks, class_names)
thresh = cm.max() / 2.0 if cm.size else 0
for i, j in np.ndindex(cm.shape):
    plt.text(j, i, f"{cm[i, j]:d}", ha='center',
            color='white' if cm[i, j] > thresh else 'black')
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'results', 'model_10', 'confusion_matrix.png'))
plt.close()

# Normalized confusion matrix (rows sum to 1)
cm_norm = cm.astype('float')
row_sums = cm_norm.sum(axis=1)[:, np.newaxis]
with np.errstate(divide='ignore', invalid='ignore'):
    cm_norm = np.divide(cm_norm, row_sums)
cm_norm = np.nan_to_num(cm_norm)

plt.figure(figsize=(8, 6))
plt.imshow(cm_norm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Normalized Confusion Matrix')
plt.colorbar()
plt.xticks(tick_marks, class_names, rotation=45, ha='right')
plt.yticks(tick_marks, class_names)
thresh = cm_norm.max() / 2.0 if cm_norm.size else 0
for i, j in np.ndindex(cm_norm.shape):
    plt.text(j, i, f"{cm_norm[i, j]:.2f}", ha='center',
            color='white' if cm_norm[i, j] > thresh else 'black')
plt.ylabel('True label')
plt.xlabel('Predicted label (normalized)')
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'results', 'model_10', 'confusion_matrix_normalized.png'))
plt.close()

# Plot training curves
plt.figure()
plt.plot(history.history.get('loss', []), label='train_loss')
plt.plot(history.history.get('val_loss', []), label='val_loss')
plt.legend()
plt.title('Loss')
plt.savefig(os.path.join(BASE_DIR, 'results', 'model_10', 'loss_curve.png'))
plt.close()

plt.figure()
plt.plot(history.history.get('accuracy', []), label='train_acc')
plt.plot(history.history.get('val_accuracy', []), label='val_acc')
plt.legend()
plt.title('Accuracy')
plt.savefig(os.path.join(BASE_DIR, 'results', 'model_10', 'accuracy_curve.png'))
plt.close()

# Save the trained model
model_save_path = os.path.join(BASE_DIR, "models", "model_10", "hand_gesture_model_10.keras")
model.save(model_save_path)
print(f"\nModel saved to {model_save_path}")

# Convert the model.
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Save the model.
with open(os.path.join(BASE_DIR, "models", "model_10", "model_10.tflite"), 'wb') as f:
    f.write(tflite_model)

3/3 [==============================] - 0s 6ms/step

Confusion Matrix:
 [[10  0  0  0  0  0  0  0]
 [ 0 10  0  0  0  0  0  0]
 [ 0  0  9  1  0  0  0  0]
 [ 0  0  0 10  0  0  0  0]
 [ 0  0  0  0 10  0  0  0]
 [ 0  0  0  0  0 10  0  0]
 [ 0  0  0  0  0  0 10  0]
 [ 0  0  0  0  0  0  0 10]]

Classification Report:
                  precision    recall  f1-score   support

     Background       1.00      1.00      1.00        10
    New Speed 1       1.00      1.00      1.00        10
    New Speed 2       1.00      0.90      0.95        10
    New Speed 3       0.91      1.00      0.95        10
    New Speed 4       1.00      1.00      1.00        10
New Stop Images       1.00      1.00      1.00        10
  Pointing Left       1.00      1.00      1.00        10
 Pointing Right       1.00      1.00      1.00        10

       accuracy                           0.99        80
      macro avg       0.99      0.99      0.99        80
   weighted avg       0.99      0.99      0.99        80



INFO:tensorflow:Assets written to: C:\Users\louis\AppData\Local\Temp\tmpumbprgiw\assets
